### SVM Theory : Soft Margin Primal Form 
 * **Hyperplane** is the decision boundary that separates classes
 * **Margin** : The perpendicular distance between the decision boundary and the nearest training points from either class. $\frac{1}{\|w\|}$
 * **Classification Condition** : All training points must lie outside or on the margin hyperplanes; ie. $y_i(w.x_i+b)≥1  \text{ for all i}$
 * **Support vectors** Training samples closest to the decision boundary.
 * **Hard margin optimization** problem : $\min_{w,b}\ \frac{1}{2}\|w\|^2 \quad $ Subject to $\quad y_i(w^\top x_i + b) \ge 1,\ \forall i$
 * **Soft margin optimization** problem : $\min_{w,b,\xi}\ \frac{1}{2}\|w\|^2 + C\sum_{i=1}^{n}\xi_i \quad $ Subject to $\quad y_i(w^\top x_i + b) \ge 1 - \xi_i,\ \forall i$ with $\xi_i \ge 0,\ \forall i$
* **Constrained to Unconstrained :** $\xi_i \ge 1 - y_i(w^\top x_i + b),\ \xi_i \ge 0 \quad \Rightarrow \quad \xi_i = \max\left(0,\ 1 - y_i(w^\top x_i + b)\right)$
* **Hinge loss** is defined by $\max\left(0,\ 1-y_i(w^\top x_i+b)\right)$
* **Softmargin Optimization Updated** : $\min_{w,b}\ \frac{1}{2}\|w\|^2 + C\sum_{i=1}^{n}\max\left(0,\ 1-y_i(w^\top x_i+b)\right)$
* **Decision Boundary**: $w^Tx+b = 0$
* $|w^Tx+b|$ measures how strong one point belongs to one side
* $w^Tx$ is projection of x on to w
* positive-class points to have scores at least +1
* negative-class points to have scores at most -1

In [63]:
import numpy as np

class SoftMarginSVM:
    def __init__(self,learning_rate=0.001,lambda_param=0.01,epochs=1000):
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.epochs = epochs
        self.w = None
        self.b = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0
        for epoch in range(self.epochs):
            for idx, x_i in enumerate(X):
                margin = y[idx] * (np.dot(x_i, self.w) + self.b)

                # Correctly classified and outside margin
                if margin >= 1:
                    dw = self.lambda_param * self.w
                    db = 0.0

                # Inside margin OR misclassified
                else:
                    dw = (self.lambda_param * self.w - y[idx] * x_i)
                    db = -y[idx]
                self.w -= self.lr * dw
                self.b -= self.lr * db
                
            # Monitor total loss after each epoch
            epoch_loss = self.total_loss(X, y)
            print(f"Epoch {epoch + 1}/{self.epochs} "f"| Total Loss: {epoch_loss:.4f}")

    def decision_function(self, X):
        return np.dot(X, self.w) + self.b

    def predict(self, X):
        scores = self.decision_function(X)
        
        return scores,np.sign(scores)

    def hinge_loss(self, X, y):
        margins = y * self.decision_function(X)
        losses = np.maximum(0, 1 - margins)
        return np.mean(losses)

    def regularization_loss(self):
        return (self.lambda_param / 2) * np.dot(self.w, self.w)

    def total_loss(self, X, y):
        return (self.regularization_loss()+ self.hinge_loss(X, y))

    def get_support_vectors(self, X, y):

        # Compute margin values
        margins = y * self.decision_function(X)
    
        # Support vectors:
        # points lying on or inside margin
        support_vector_indices = np.where(margins <= 1)[0]
    
        # Extract vectors and labels
        support_vectors = X[support_vector_indices]
        support_vector_labels = y[support_vector_indices]
    
        return (
            support_vector_indices,
            support_vectors,
            support_vector_labels
        )

In [81]:
np.random.seed(42)

# Positive class centered around +1
X_pos = np.random.randn(20, 4) + 1

# Negative class centered around 0
X_neg = np.random.randn(20, 4) -9

# Combine
X = np.vstack((X_pos, X_neg))

# Labels
y = np.hstack((np.ones(20),-np.ones(20)))
X.shape,y.shape

((40, 4), (40,))

In [82]:
svm = SoftMarginSVM(
    learning_rate=0.001,
    lambda_param=0.01,
    epochs=10
)

svm.fit(X, y)

Epoch 1/10 | Total Loss: 0.4281
Epoch 2/10 | Total Loss: 0.3866
Epoch 3/10 | Total Loss: 0.3451
Epoch 4/10 | Total Loss: 0.3036
Epoch 5/10 | Total Loss: 0.2621
Epoch 6/10 | Total Loss: 0.2207
Epoch 7/10 | Total Loss: 0.1883
Epoch 8/10 | Total Loss: 0.1675
Epoch 9/10 | Total Loss: 0.1518
Epoch 10/10 | Total Loss: 0.1363


In [83]:
test_sample = X[0]
test_sample

array([1.49671415, 0.8617357 , 1.64768854, 2.52302986])

In [84]:
svm.predict(test_sample)

(np.float64(1.3332849682407022), np.float64(1.0))

In [85]:
svm.predict(X)

(array([ 1.33328497,  1.1578383 ,  0.69262461,  0.26657874,  0.30244034,
         0.8110691 ,  0.6933741 ,  1.00377097,  0.56255718,  0.4649173 ,
         0.94207755,  0.63474107,  0.62737363,  1.15917849,  0.90570463,
         0.34391188,  1.40545032,  1.19709947,  0.89033401,  0.43402232,
        -5.7905824 , -5.92900139, -5.709222  , -6.43381361, -5.85512809,
        -6.43571245, -5.57369516, -6.13498467, -5.49027087, -5.76645997,
        -5.98403727, -5.72285061, -6.15687062, -5.83389094, -6.23324037,
        -5.84999784, -6.1804229 , -5.65463084, -6.09798406, -5.5168167 ]),
 array([ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
         1.,  1.,  1.,  1.,  1.,  1.,  1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1.]))

In [86]:
np.mean(y==svm.predict(X)[1]) # accuracy

np.float64(1.0)

In [87]:
len(svm.get_support_vectors(X,y)[0])

14